# RAG Assignment: Baseline Retrieval vs. Cross-Encoder Re-Ranking

Two retrieval pipelines are run over the **same chunks** of one PDF (the ICC Playing Handbook, cricket) and compared:

| | Pipeline 1 (Baseline) | Pipeline 2 (Re-Ranking) |
|---|---|---|
| Stage 1 | Bi-Encoder + cosine similarity, Top 10 | same Top 10 |
| Stage 2 | take Top 3 directly | Cross-Encoder re-scores the 10, take Top 3 |
| Generation | local LLM (Ollama), prompt P | the **same** prompt P |

**Sections:** 1 Setup, 2 Load and chunk, 3 Index, 4 Pipelines, 5 LLM, 6 Compare helper,
7 Failure-mode tests, 8 Latency, 9 Notes for the report.

## 1. Setup and configuration
All tunable settings live in one cell, so every experiment can be reproduced from it.

In [3]:
import json
import os
import re
import time
from pathlib import Path

os.environ["TOKENIZERS_PARALLELISM"] = "false"   # silence a harmless tokenizer warning

import faiss
import numpy as np
import ollama
import pandas as pd
import pymupdf                                  # PDF reader that can read a part of a page (needed for this two-page-spread PDF)
from sentence_transformers import CrossEncoder, SentenceTransformer

# ---- Configuration -----------------------------------------------------------
PDF_PATH = Path("data/cricket_rulebook.pdf")   # the single PDF used for the whole assignment

BI_ENCODER_NAME    = "sentence-transformers/all-MiniLM-L6-v2"   # dense embeddings (Bi-Encoder)
CROSS_ENCODER_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"     # re-ranker (Cross-Encoder)
LLM_NAME           = "llama3.1:8b"                              # local model served by Ollama

CHUNK_WORDS   = 150    # chunk length in words (about 200 tokens, fits the 256-token limit of the Bi-Encoder)
OVERLAP_WORDS = 30     # words shared by neighbouring chunks so a sentence is not cut in half
# ---- PDF clean-up settings (specific to this document, see Section 2) ----
SKIP_PDF_PAGES = {1}                    # cover; the other front/back matter is dropped by the contents / too-short rules
MIN_PAGE_WORDS = 20                     # half-pages with fewer words are blank / dividers, not content
HEADER_BOTTOM = 40                      # y (in points) where the running header ends; text above is the header
FOOTER_TOP = 550                        # y where the printed page number (folio, e.g. "3.30") starts

# ---- Saving the cleaned data (so the PDF is not re-processed on every run) ----
PROCESSED_DIR = Path("data/processed/cricket")  # cleaned pages and chunks are saved here
USE_CACHE = True                        # True: reuse saved files if the settings above did not change
CLEANING_VERSION = "cricket-v3-clauses"    # change this text whenever clean_page() is edited, to force a rebuild

TOP_K_RETRIEVE = 10    # candidates fetched by the Bi-Encoder
TOP_K_CONTEXT  = 3     # chunks given to the LLM

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 200)

## 2. Load, clean and chunk the PDF
Both pipelines search the exact same list of chunks. The document is the **ICC Playing Handbook 2019-20**
(427 PDF pages, about 300,000 words). It is hard to read as text, so cleaning matters. The cleaning steps are:

| Problem in the raw PDF | What we do |
|---|---|
| Every PDF page is a **spread** of two printed pages side by side (two columns). Reading the page in one go mixes the columns | read the **left half and the right half separately** (`pymupdf` with a clip rectangle) = 854 half-pages in the right order |
| Running header (part name) and the printed page number (`3.30`) on every half-page | cut by position (top band = header, bottom band = page number). The header becomes the `section` label, the page number becomes `page` |
| The header is spelled in many ways (with/without "PLAYING CONDITIONS", `CONTENTS` pages) | map every header to one of 12 fixed part names (`canonical_part`) |
| Contents lists, cover, blank pages | drop pages that start with `CONTENTS`, or are mostly page numbers, or have fewer than `MIN_PAGE_WORDS` words |
| Form lines (`________`), dot leaders, ligatures, odd spaces, curly quotes, a private-use tick glyph | normalise to plain text; bullets are kept as ` • ` |
| The five formats (Men's Test, ODI, T20I, Women's Championship, Women's T20I) repeat the same clauses with small changes | the part name is **written at the start of every chunk** (`[ICC Men's One Day International Playing Conditions] ...`), so the embedding can tell the formats apart |
| Fixed windows cut clauses in half | the text is split at **clause numbers** (`12.9.1`, `(a)`) and whole clauses are packed into a chunk of at most `CHUNK_WORDS` words; the last clause(s) are repeated at the start of the next chunk (`OVERLAP_WORDS`). A clause longer than the limit is cut at sentence ends |
| Tables (powerplay table, logo sizes) | stay as text; the numbers read like a list. This is a known limit, only about 1% of chunks |

Each chunk remembers the page and the part where it starts. Chunking is done **inside one part only**, so a chunk never mixes two formats.

The cleaned pages and chunks are saved in `data/processed/cricket/` (JSON for the notebook, `.txt` and `.csv` for reading).
On the next run they are loaded from there, unless the settings changed.

In [4]:
FOLIO_RE = re.compile(r"^\d{1,2}\.\d{1,3}$")                       # printed page number such as "3.30"
# A clause number at the start of a line: "12.9.1" alone, or "12.8<TAB> Extra Time", or "(a)"
CLAUSE_RE = re.compile(r"^(\d{1,3}(?:\.\d{1,3}){0,6}|\([a-z0-9]{1,4}\))(\t|\s*$)")
SENTENCE_SPLIT_RE = re.compile(r"(?<=[.;:])\s+(?=[A-Z\u2022(])")

DEFAULT_PART = "ICC Playing Handbook (general information)"   # front pages before the first part header
PART_RULES = [   # (words that must all appear in the header, part name)
    (("MEN'S", "WORLD TEST"), "ICC Men's World Test Championship Playing Conditions"),
    (("MEN'S", "ONE DAY"), "ICC Men's One Day International Playing Conditions"),
    (("MEN'S", "TWENTY20"), "ICC Men's Twenty20 International Playing Conditions"),
    (("WOMEN'S", "CHAMPIONSHIP"), "ICC Women's Championship Playing Conditions"),
    (("WOMEN'S", "TWENTY20"), "ICC Women's Twenty20 International Playing Conditions"),
    (("CODE OF CONDUCT",), "ICC Code of Conduct for Players and Player Support Personnel"),
    (("DISCRIMINATION",), "ICC Anti-Discrimination Code for Participants"),
    (("CORRUPTION",), "ICC Anti-Corruption Code for Participants"),
    (("REVIEW OF BOWLERS",), "ICC Regulations for the Review of Bowlers Reported with Suspected Illegal Bowling Actions"),
    (("CLOTHING",), "ICC Clothing and Equipment Rules and Regulations"),
    (("SAFEGUARDING",), "ICC Safeguarding Regulations"),
    (("OTHER ICC",), "Other ICC Regulations"),
]


def canonical_part(header_text):
    """Map the many header spellings to one fixed part name (or None if the header is empty / unknown)."""
    t = header_text.replace("\u2019", "'").upper()
    for needles, name in PART_RULES:
        # whole-word match: "MEN'S" must not match inside "WOMEN'S"
        if all(re.search(r"(?<![A-Z])" + re.escape(n), t) for n in needles):
            return name
    return None


def read_halves(pdf_path):
    """Each PDF page is a spread (left page + right page). Read each half on its own so the order is correct.
    Returns one dict per half-page with its header lines, footer lines and body lines."""
    if not Path(pdf_path).exists():
        raise FileNotFoundError(f"PDF not found at '{pdf_path}'. Put the document there or change PDF_PATH.")
    doc = pymupdf.open(str(pdf_path))
    halves = []
    for pdf_page, page in enumerate(doc, start=1):
        w, h = page.rect.width, page.rect.height
        for side, x0, x1 in (("L", 0, w / 2), ("R", w / 2, w)):
            def lines(y0, y1):
                return page.get_text("text", clip=pymupdf.Rect(x0, y0, x1, y1)).splitlines()
            halves.append({"pdf_page": pdf_page, "side": side,
                           "header": [l.strip() for l in lines(0, HEADER_BOTTOM) if l.strip()],
                           "footer": [l.strip() for l in lines(FOOTER_TOP, h) if l.strip()],
                           "body_lines": lines(HEADER_BOTTOM, FOOTER_TOP)})
    return halves


def clean_text(text):
    text = text.replace("\t", " ").replace("\u00a0", " ").replace("\ufb01", "fi").replace("\ufb02", "fl")
    text = text.replace("\uf0fc", "\u2022")                                # private-use tick glyph -> bullet
    text = text.replace("\u2019", "'").replace("\u2018", "'").replace("\u201c", '"').replace("\u201d", '"')
    text = re.sub(r"_{3,}", "___", text)                                    # form fill-in lines
    text = re.sub(r"\.{4,}", "...", text)                                   # dot leaders
    text = re.sub(r"\s*\u2022\s*", " \u2022 ", text)                        # bullet with a space on each side
    return re.sub(r"\s+", " ", text).strip()


def is_contents(body_lines):
    """A contents list: starts with CONTENTS, or is mostly '<title> 5.3' lines full of page numbers."""
    lines = [l.strip() for l in body_lines if l.strip()]
    if not lines:
        return False
    if lines[0].upper() == "CONTENTS":
        return True
    words = " ".join(lines).split()
    return sum(bool(FOLIO_RE.match(w)) for w in words) / len(words) > 0.12


def clean_pdf(pdf_path, skip_pdf_pages, min_words):
    """Returns (pages, dropped). pages: one dict per kept half-page with
    {"pdf_page", "side", "page" (printed number), "section" (part name), "text", "units" (clauses)}.
    dropped: list of [pdf_page, side, reason]."""
    pages, dropped, part = [], [], DEFAULT_PART
    for h in read_halves(pdf_path):
        header_words = [w for line in h["header"] for w in line.split() if not re.fullmatch(r"\d{1,2}", w)]
        part = canonical_part(" ".join(header_words)) or part      # an empty header keeps the previous part
        body_text = clean_text("\n".join(h["body_lines"]))
        reason = None
        if h["pdf_page"] in skip_pdf_pages:
            reason = "skipped"
        elif is_contents(h["body_lines"]):
            reason = "contents"
        elif len(body_text.split()) < min_words:
            reason = "too short"
        if reason:
            dropped.append([h["pdf_page"], h["side"], reason])
            continue
        # split the body into clauses: a new unit starts at every line that begins with a clause number
        units, cur = [], []
        for line in h["body_lines"]:
            if CLAUSE_RE.match(line.strip()) and cur:
                units.append(cur); cur = []
            cur.append(line)
        if cur:
            units.append(cur)
        folio = next((l for l in h["footer"] if FOLIO_RE.match(l)), None)
        pages.append({"pdf_page": h["pdf_page"], "side": h["side"], "page": folio or f"{h['pdf_page']}{h['side']}",
                      "section": part, "text": body_text,
                      "units": [u for u in (clean_text("\n".join(x)) for x in units) if u]})
    return pages, dropped


def split_long(unit, max_words):
    """A clause longer than max_words is cut at sentence ends and, only if still too long, by words."""
    if len(unit.split()) <= max_words:
        return [unit]
    out = []
    for sentence in SENTENCE_SPLIT_RE.split(unit):
        words = sentence.split()
        for k in range(0, len(words), max_words):
            out.append(" ".join(words[k:k + max_words]))
    return out


def make_chunks(pages, chunk_words, overlap_words):
    """Per part: pack whole clauses into chunks of at most chunk_words words. The last clauses (about
    overlap_words words) are repeated at the start of the next chunk. The part name starts every chunk."""
    groups, current = [], []
    for p in pages:                                   # consecutive pages with the same part form one group
        if current and current[-1]["section"] != p["section"]:
            groups.append(current); current = []
        current.append(p)
    if current:
        groups.append(current)
    chunks = []
    for group in groups:
        label = group[0]["section"]
        units = [(piece, p["page"], len(piece.split()))                       # (text, page, n_words)
                 for p in group for u in p["units"] for piece in split_long(u, chunk_words)]
        i = 0
        while i < len(units):
            j, count = i, 0
            while j < len(units) and (count == 0 or count + units[j][2] <= chunk_words):
                count += units[j][2]; j += 1
            chunks.append({"chunk_id": len(chunks), "page": units[i][1], "section": label,
                           "text": f"[{label}] " + " ".join(u[0] for u in units[i:j])})
            if j >= len(units):
                break
            k, shared = j, 0                          # step back over the last clauses to create the overlap
            while k > i + 1 and shared + units[k - 1][2] <= overlap_words:
                k -= 1; shared += units[k][2]
            i = k
    return chunks


def save_processed(folder, meta, pages, dropped, chunks):
    """Save cleaned pages and chunks: JSON (reloaded by this notebook), TXT and CSV (for reading)."""
    folder.mkdir(parents=True, exist_ok=True)
    dump = lambda obj: json.dumps(obj, indent=1, ensure_ascii=False)
    (folder / "cleaned_pages.json").write_text(dump({"meta": meta, "dropped_pages": dropped, "pages": pages}), encoding="utf-8")
    (folder / "chunks.json").write_text(dump({"meta": meta, "chunks": chunks}), encoding="utf-8")
    with open(folder / "cleaned_document.txt", "w", encoding="utf-8") as f:
        for p in pages:
            f.write(f"=== Page {p['page']} (PDF page {p['pdf_page']}{p['side']}) | {p['section']} ===\n{p['text']}\n\n")
    table = pd.DataFrame(chunks)
    table.insert(3, "words", table["text"].str.split().str.len())
    table.to_csv(folder / "chunks.csv", index=False, encoding="utf-8")


def load_processed(folder, meta):
    """Return (pages, dropped, chunks) from the saved files, or None if missing or made with other settings."""
    try:
        saved_pages = json.loads((folder / "cleaned_pages.json").read_text(encoding="utf-8"))
        saved_chunks = json.loads((folder / "chunks.json").read_text(encoding="utf-8"))
    except FileNotFoundError:
        return None
    if saved_pages["meta"] != meta or saved_chunks["meta"] != meta:
        return None
    return saved_pages["pages"], saved_pages["dropped_pages"], saved_chunks["chunks"]

In [5]:
# Settings that decide what the cleaned data looks like. If any of them changes, the saved files are rebuilt.
meta = {
    "pdf": PDF_PATH.name, "pdf_bytes": PDF_PATH.stat().st_size,
    "chunk_words": CHUNK_WORDS, "overlap_words": OVERLAP_WORDS,
    "skip_pdf_pages": sorted(SKIP_PDF_PAGES), "min_page_words": MIN_PAGE_WORDS,
    "header_bottom": HEADER_BOTTOM, "footer_top": FOOTER_TOP,
    "cleaning_version": CLEANING_VERSION,
}

cached = load_processed(PROCESSED_DIR, meta) if USE_CACHE else None
if cached:
    pages, dropped_pages, chunks = cached
    print(f"Loaded cleaned pages and chunks from {PROCESSED_DIR}/ (set USE_CACHE = False to rebuild)")
else:
    pages, dropped_pages = clean_pdf(PDF_PATH, SKIP_PDF_PAGES, MIN_PAGE_WORDS)
    chunks = make_chunks(pages, CHUNK_WORDS, OVERLAP_WORDS)
    save_processed(PROCESSED_DIR, meta, pages, dropped_pages, chunks)
    print(f"Cleaned the PDF and saved the results to {PROCESSED_DIR}/")

chunk_texts = [c["text"] for c in chunks]

half_pages = len(pages) + len(dropped_pages)
print(f"{half_pages} half-pages -> kept {len(pages)}, dropped {len(dropped_pages)} (cover / contents / blank)")
print(f"{sum(len(p['text'].split()) for p in pages):,} words  ->  {len(chunks)} chunks")
print("\nChunks per part:")
print(pd.Series([c["section"] for c in chunks]).value_counts().to_string())

# Chunks longer than the Bi-Encoder limit (256 tokens) are cut off silently, so measure how many there are.
_tok = SentenceTransformer(BI_ENCODER_NAME).tokenizer
_lens = [len(_tok(t)["input_ids"]) for t in chunk_texts]
print(f"\nTokens per chunk: mean {np.mean(_lens):.0f}, max {max(_lens)}; {sum(l > 256 for l in _lens)} chunks are over 256 tokens (mostly number tables)")
print("\nExample chunk 365:", chunks[365]["section"], "| page", chunks[365]["page"], "\n", chunks[365]["text"][:400], "...")

Loaded cleaned pages and chunks from data/processed/cricket/ (set USE_CACHE = False to rebuild)
854 half-pages -> kept 812, dropped 42 (cover / contents / blank)
296,787 words  ->  2660 chunks

Chunks per part:
ICC Men's World Test Championship Playing Conditions                                         400
ICC Women's Twenty20 International Playing Conditions                                        388
ICC Women's Championship Playing Conditions                                                  384
ICC Men's One Day International Playing Conditions                                           383
ICC Men's Twenty20 International Playing Conditions                                          383
ICC Code of Conduct for Players and Player Support Personnel                                 182
ICC Clothing and Equipment Rules and Regulations                                             154
ICC Anti-Corruption Code for Participants                                                    122
ICC Regulatio

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (302 > 256). Running this sequence through the model will result in indexing errors



Tokens per chunk: mean 168, max 359; 17 chunks are over 256 tokens (mostly number tables)

Example chunk 365: ICC Men's World Test Championship Playing Conditions | page 3.109 
 [ICC Men's World Test Championship Playing Conditions] 3.2.3 The captain may consult with the bowler and other fielders, and the two batsmen may consult with each other prior to deciding whether to request a Player Review. Under no circumstances is any player permitted to query an umpire about any aspect of a decision before deciding on whether or not to request a Player Review. If the on-field um ...


## 3. Index the chunks (Bi-Encoder + vector store)
Each chunk is turned into one vector **independently** of any query. Vectors are L2-normalised,
so the inner product equals cosine similarity. `IndexFlatIP` does an exact (not approximate) search.

In [6]:
bi_encoder = SentenceTransformer(BI_ENCODER_NAME)

chunk_vectors = bi_encoder.encode(
    chunk_texts, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=True
).astype("float32")

index = faiss.IndexFlatIP(chunk_vectors.shape[1])
index.add(chunk_vectors)
print("Indexed vectors:", index.ntotal, "| dimension:", chunk_vectors.shape[1])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/84 [00:00<?, ?it/s]

Indexed vectors: 2660 | dimension: 384


## 4. The two retrieval pipelines

* **Pipeline 1** keeps the Bi-Encoder order and uses its Top 3.
* **Pipeline 2** starts from the same Top 10, scores every `(query, chunk)` pair with the Cross-Encoder,
  re-sorts by that score, and keeps the Top 3.

In [7]:
cross_encoder = CrossEncoder(CROSS_ENCODER_NAME)


def retrieve_top_k(query, k=TOP_K_RETRIEVE):
    """Pipeline 1, stage 1: dense retrieval. Returns [(chunk_id, cosine_score), ...] best first."""
    q_vec = bi_encoder.encode([query], normalize_embeddings=True, convert_to_numpy=True).astype("float32")
    scores, ids = index.search(q_vec, min(k, index.ntotal))     # never ask for more than exist
    return [(int(i), float(s)) for i, s in zip(ids[0], scores[0])]


def rerank(query, candidates):
    """Pipeline 2, stage 2: score each (query, chunk) pair jointly, sort high to low.
    candidates is the output of retrieve_top_k. Returns [(chunk_id, cross_encoder_score), ...]."""
    pairs = [(query, chunks[cid]["text"]) for cid, _ in candidates]
    ce_scores = cross_encoder.predict(pairs)
    reranked = sorted(zip([cid for cid, _ in candidates], ce_scores), key=lambda x: x[1], reverse=True)
    return [(cid, float(s)) for cid, s in reranked]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

## 5. Answer generation (local LLM through Ollama)
The prompt is identical for both pipelines, and `temperature=0` makes the output (almost) deterministic,
so any difference between Answer 1 and Answer 2 comes from the retrieved chunks alone.

In [8]:
SYSTEM_PROMPT = (
    "You answer questions about a document using ONLY the excerpts provided. "
    "If the excerpts do not contain the answer, say \"The provided excerpts do not contain the answer.\" "
    "Mention any exception or restriction that appears in the excerpts. Be concise."
)


def build_prompt(query, chunk_ids):
    """Same prompt structure for both pipelines: numbered excerpts, then the question."""
    excerpts = "\n\n".join(
        f"[Excerpt {n}] (page {chunks[cid]['page']}, {chunks[cid]['section']})\n{chunks[cid]['text']}"
        for n, cid in enumerate(chunk_ids, start=1)
    )
    return f"{excerpts}\n\nQuestion: {query}\nAnswer:"


def generate_answer(query, chunk_ids):
    response = ollama.chat(
        model=LLM_NAME,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": build_prompt(query, chunk_ids)},
        ],
        options={"temperature": 0, "num_ctx": 4096},
    )
    return response["message"]["content"].strip()


# Warm-up: loads the LLM into memory so the first real query is not slower than the rest.
_ = ollama.chat(model=LLM_NAME, messages=[{"role": "user", "content": "hi"}], options={"num_predict": 1})

## 6. Side-by-side comparison helper
`compare(query, gold_phrase=...)` runs both pipelines and shows:
1. the **rank shift table** for the 10 candidates (Bi-Encoder rank -> Cross-Encoder rank),
2. **Answer 1** (baseline) and **Answer 2** (re-ranked),
3. stage timings.

`gold_phrase` is optional: a short piece of text copied from the PDF that the correct chunk must contain.
Chunks that contain it are marked as gold, which makes rank movements easy to read.

In [9]:
def _norm(text):
    # Drop ALL whitespace: PDF extraction sometimes inserts stray spaces inside words.
    return re.sub(r"\s+", "", text.lower())


def find_gold_chunks(gold_phrase):
    """Chunk ids whose text contains the phrase (case- and whitespace-insensitive)."""
    if not gold_phrase:
        return set()
    needle = _norm(gold_phrase)
    return {c["chunk_id"] for c in chunks if needle in _norm(c["text"])}


def compare(query, gold_phrase=None, show_answers=True):
    gold = find_gold_chunks(gold_phrase)
    if gold_phrase and not gold:
        print(f"WARNING: gold_phrase not found in any chunk: {gold_phrase!r}")

    # ---- Pipeline 1: Bi-Encoder only ----
    t0 = time.perf_counter()
    candidates = retrieve_top_k(query)
    t_retrieve = time.perf_counter() - t0
    base_top = [cid for cid, _ in candidates[:TOP_K_CONTEXT]]

    # ---- Pipeline 2: Cross-Encoder on the same candidates ----
    t0 = time.perf_counter()
    reranked = rerank(query, candidates)
    t_rerank = time.perf_counter() - t0
    rerank_top = [cid for cid, _ in reranked[:TOP_K_CONTEXT]]

    # ---- Rank shift table ----
    bi_rank = {cid: r for r, (cid, _) in enumerate(candidates, start=1)}
    bi_score = dict(candidates)
    ce_rank = {cid: r for r, (cid, _) in enumerate(reranked, start=1)}
    ce_score = dict(reranked)
    table = pd.DataFrame([{
        "chunk_id": cid,
        "page": chunks[cid]["page"],
        "section": (chunks[cid]["section"] or "")[:22],
        "bi_rank": bi_rank[cid],
        "ce_rank": ce_rank[cid],
        "shift": bi_rank[cid] - ce_rank[cid],            # positive = moved up
        "bi_score": round(bi_score[cid], 3),
        "ce_score": round(ce_score[cid], 2),
        "gold": "GOLD" if cid in gold else "",
        "preview": chunks[cid]["text"][:90],
    } for cid, _ in candidates]).sort_values("ce_rank")

    # ---- LLM answers ----
    answer1 = answer2 = None
    t_llm1 = t_llm2 = 0.0
    if show_answers:
        t0 = time.perf_counter(); answer1 = generate_answer(query, base_top);   t_llm1 = time.perf_counter() - t0
        t0 = time.perf_counter(); answer2 = generate_answer(query, rerank_top); t_llm2 = time.perf_counter() - t0

    print("QUERY:", query)
    if gold:
        print("Gold chunk(s):", sorted(gold),
              "| baseline ranks:", [bi_rank.get(g, ">10") for g in sorted(gold)],
              "| re-ranked ranks:", [ce_rank.get(g, ">10") for g in sorted(gold)])
    display(table.reset_index(drop=True))
    if show_answers:
        print(f"\n--- Answer 1 (baseline, chunks {base_top}) ---\n{answer1}")
        print(f"\n--- Answer 2 (re-ranked, chunks {rerank_top}) ---\n{answer2}")
    print(f"\nTimings: retrieval {t_retrieve*1000:.1f} ms | re-rank {t_rerank*1000:.1f} ms | "
          f"LLM {t_llm1:.1f}s / {t_llm2:.1f}s")
    return {"query": query, "gold": gold, "table": table, "answer1": answer1, "answer2": answer2,
            "base_top": base_top, "rerank_top": rerank_top}

### Quick test
Run one question through both pipelines. Change `query` and `gold_phrase` to try your own.
`gold_phrase` is a short piece of text that the correct chunk must contain (leave it out if you do not know it yet).

In [10]:
query = "During the Powerplay in a Men's T20 international, how many fielders can be outside the fielding restriction area?"
gold_phrase = "only two fielders"

_ = compare(query, gold_phrase)

QUERY: During the Powerplay in a Men's T20 international, how many fielders can be outside the fielding restriction area?
Gold chunk(s): [964, 965, 1735] | baseline ranks: [10, 1, 2] | re-ranked ranks: [6, 1, 2]


,chunk_id,page,section,bi_rank,ce_rank,shift,bi_score,ce_score,gold,preview
0,965,5.54,ICC Men's Twenty20 Int,1,1,0,0.773,8.31,GOLD,[ICC Men's Twenty20 International Playing Conditions] 28.7.4 During the Powerplay over...
1,1735,9.56,ICC Women's Twenty20 I,2,2,0,0.767,7.46,GOLD,[ICC Women's Twenty20 International Playing Conditions] 28.7.4 During the Powerplay ov...
2,589,4.55,ICC Men's One Day Inte,3,3,0,0.746,7.34,,[ICC Men's One Day International Playing Conditions] 28.7.2.2.2 Powerplay 2 – no more ...
3,1356,8.56,ICC Women's Championsh,4,4,0,0.728,5.94,,"[ICC Women's Championship Playing Conditions] 28.7.2.3 During non Powerplay overs, no ..."
4,1354,8.55,ICC Women's Championsh,5,5,0,0.727,4.58,,[ICC Women's Championship Playing Conditions] 28.7.2.2 Subject to the provisions of 28...
5,964,5.54,ICC Men's Twenty20 Int,10,6,4,0.659,3.59,GOLD,[ICC Men's Twenty20 International Playing Conditions] 28.7.2 Subject to 28.7.6 below t...
6,963,5.54,ICC Men's Twenty20 Int,6,7,-1,0.693,1.49,,[ICC Men's Twenty20 International Playing Conditions] 28.6.2 In all circumstances clau...
7,959,5.53,ICC Men's Twenty20 Int,8,8,0,0.671,-0.34,,[ICC Men's Twenty20 International Playing Conditions] 28.3.4 If the ball while in play...
8,1733,9.56,ICC Women's Twenty20 I,7,9,-2,0.692,-0.61,,[ICC Women's Twenty20 International Playing Conditions] 28.6.2 In all circumstances cl...
9,1729,9.55,ICC Women's Twenty20 I,9,10,-1,0.670,-2.78,,[ICC Women's Twenty20 International Playing Conditions] 28.3.4 If the ball while in pl...



--- Answer 1 (baseline, chunks [965, 1735, 589]) ---
According to Excerpt 1, during the Powerplay overs, only two fielders shall be permitted outside the fielding restriction area at the instant of delivery (clause 28.7.4).

--- Answer 2 (re-ranked, chunks [965, 1735, 589]) ---
According to Excerpt 1, during the Powerplay overs, only two fielders shall be permitted outside the fielding restriction area at the instant of delivery (clause 28.7.4).

Timings: retrieval 11.1 ms | re-rank 43.1 ms | LLM 2.2s / 2.0s


## 7. Failure-mode stress tests
Each test is a dictionary with a `category`, a `query` and a `gold_phrase`.
Categories required by the assignment: **Keyword trap**, **Exception / corner case**,
**Negation / constraint**, **Buried chunk (rank 4-10 promoted to Top 3)**.
At least two queries must show the baseline failing and the re-ranker fixing it.

> The list is filled in after the PDF is chosen and the document has been explored.

In [11]:
FAILURE_QUERIES = []
# Template for each entry:
# {"category": "Keyword trap", "query": "...", "gold_phrase": "exact words from the PDF that the right chunk contains"}

In [12]:
results = []
for item in FAILURE_QUERIES:
    print("=" * 110)
    print("CATEGORY:", item["category"])
    results.append(compare(item["query"], item.get("gold_phrase")))
if not FAILURE_QUERIES:
    print("FAILURE_QUERIES is empty - add queries after the PDF is chosen.")

FAILURE_QUERIES is empty - add queries after the PDF is chosen.


## 8. Latency profiling
Each stage is timed separately with `time.perf_counter()` over several repetitions
(after one warm-up run), and the mean and standard deviation are reported.
The Cross-Encoder time is the **extra** cost that Pipeline 2 adds on top of Pipeline 1.

In [13]:
def profile_latency(queries, n_repeats=10, include_llm=True):
    """Mean/std latency of each stage over `queries` x `n_repeats` runs."""
    t_ret, t_rr, t_llm1, t_llm2 = [], [], [], []
    for q in queries:
        retrieve_top_k(q); rerank(q, retrieve_top_k(q))            # warm-up (not timed)
        for _ in range(n_repeats):
            t0 = time.perf_counter(); cands = retrieve_top_k(q);   t_ret.append(time.perf_counter() - t0)
            t0 = time.perf_counter(); rr = rerank(q, cands);       t_rr.append(time.perf_counter() - t0)
        if include_llm:
            t0 = time.perf_counter(); generate_answer(q, [c for c, _ in cands[:TOP_K_CONTEXT]]); t_llm1.append(time.perf_counter() - t0)
            t0 = time.perf_counter(); generate_answer(q, [c for c, _ in rr[:TOP_K_CONTEXT]]);    t_llm2.append(time.perf_counter() - t0)

    ms = lambda xs: f"{np.mean(xs)*1000:.1f} +/- {np.std(xs)*1000:.1f}"
    rows = [
        ["Stage 1: Bi-Encoder retrieval (embed query + FAISS Top 10), both pipelines", ms(t_ret)],
        ["Stage 2: Cross-Encoder re-rank of 10 pairs, Pipeline 2 only",                ms(t_rr)],
        ["Pipeline 1 retrieval total (Stage 1)",                                       ms(t_ret)],
        ["Pipeline 2 retrieval total (Stage 1 + Stage 2)",                             ms(np.add(t_ret, t_rr))],
    ]
    if include_llm:
        rows += [["Pipeline 1 LLM generation", ms(t_llm1)], ["Pipeline 2 LLM generation", ms(t_llm2)]]
    return pd.DataFrame(rows, columns=["Stage", "Time in ms (mean +/- std)"])


latency_queries = [item["query"] for item in FAILURE_QUERIES] or ["What is this document about?"]
display(profile_latency(latency_queries, n_repeats=10))

,Stage,Time in ms (mean +/- std)
0,"Stage 1: Bi-Encoder retrieval (embed query + FAISS Top 10), both pipelines",5.3 +/- 0.4
1,"Stage 2: Cross-Encoder re-rank of 10 pairs, Pipeline 2 only",32.3 +/- 0.7
2,Pipeline 1 retrieval total (Stage 1),5.3 +/- 0.4
3,Pipeline 2 retrieval total (Stage 1 + Stage 2),37.6 +/- 1.0
4,Pipeline 1 LLM generation,3292.5 +/- 0.0
5,Pipeline 2 LLM generation,2767.8 +/- 0.0


## 9. Notes for the report
Fill these in after running Section 7 and 8 (own words, concise):

1. **Architecture comparison** - Bi-Encoder vs Cross-Encoder.
2. **Rank shifts and answer comparison** - per failure query: which chunk moved from rank X to rank Y, and did Answer 1 hallucinate, omit a clause or misstate a rule?
3. **Latency table** - copy from Section 8.
4. **Scaling question** - why the Cross-Encoder is not run on the whole corpus.